# Photonic Processor Research — Field Analysis

**A systematic analysis of academic literature on photonic/optical computing processors**

This notebook walks through four levels of analysis:

| Level | Focus |
|-------|-------|
| **Level 1** | Paper inventory — what's in the dataset and how relevant is each paper? |
| **Level 2** | Cluster analysis — grouping papers into subfields |
| **Level 3** | Field architecture — how are subfields connected? |
| **Level 4** | Field-engineering gaps — where are the missing links? |

**Data source:** OpenAlex API  
**Queries used:** photonic processor, photonic computing, optical computing, photonic neural network, optical neural network, photonic accelerator, optical accelerator, integrated photonic processor, silicon photonic processor, photonic matrix multiplication

---
## Level 0 — Data Download
### Fetch papers from OpenAlex API and save to CSV

> **Run this only once.** If you already have the data in `photonic_processor_data/processed/`, skip to Level 1.

In [ ]:
import requests
import json
import csv
import time
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────
OUTPUT_DIR    = Path('photonic_processor_data')
RAW_DIR       = OUTPUT_DIR / 'raw'
PROCESSED_DIR = OUTPUT_DIR / 'processed'

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = 'https://api.openalex.org/works'
EMAIL    = 'your@email.com'  # <- Add your email for higher rate limit

# Search queries
TOPIC_QUERIES = {
    'photonic_processor': [
        'photonic processor',
        'photonic computing',
        'optical computing',
        'photonic neural network',
        'optical neural network',
        'photonic accelerator',
        'optical accelerator',
        'integrated photonic processor',
        'silicon photonic processor',
        'photonic matrix multiplication',
    ]
}

FIELDS = (
    'id,title,publication_year,cited_by_count,'
    'authorships,topics,referenced_works,abstract_inverted_index'
)

print('Configuration ready.')
print(f'Output directory: {OUTPUT_DIR.resolve()}')

In [ ]:
# ── Download Function ──────────────────────────────────────────
def fetch_works(query: str, topic: str, max_results: int = 500) -> list:
    """Fetch works from OpenAlex for a given query."""
    works = []
    cursor = '*'
    params = {
        'search': query,
        'filter': 'publication_year:2000-2025',
        'select': FIELDS,
        'per-page': 200,
        'mailto': EMAIL,
    }
    print(f'  -> Query: "{query}"')
    while cursor and len(works) < max_results:
        params['cursor'] = cursor
        try:
            r = requests.get(BASE_URL, params=params, timeout=30)
            r.raise_for_status()
            data = r.json()
            results = data.get('results', [])
            if not results:
                break
            for w in results:
                w['_topic'] = topic
                w['_query'] = query
            works.extend(results)
            cursor = data.get('meta', {}).get('next_cursor')
            print(f'    {len(works)} papers loaded ...')
            time.sleep(0.15)
        except Exception as e:
            print(f'    ERROR: {e}')
            break
    return works


# ── Main Download Loop ─────────────────────────────────────────
all_works = []
seen_ids  = set()

for topic, queries in TOPIC_QUERIES.items():
    print(f"\n{'='*50}")
    print(f'Topic: {topic}')
    print(f"{'='*50}")

    topic_works = []
    for query in queries:
        fetched = fetch_works(query, topic, max_results=300)
        for w in fetched:
            wid = w.get('id', '')
            if wid and wid not in seen_ids:
                seen_ids.add(wid)
                topic_works.append(w)

    raw_file = RAW_DIR / f'{topic}_raw.jsonl'
    with open(raw_file, 'w', encoding='utf-8') as f:
        for w in topic_works:
            f.write(json.dumps(w, ensure_ascii=False) + '\n')

    print(f'  checkmark {len(topic_works)} unique papers -> {raw_file}')
    all_works.extend(topic_works)

print(f'\nTotal: {len(all_works)} unique papers across all queries')

In [ ]:
# ── Process to CSV ─────────────────────────────────────────────
print('Processing to CSVs ...')

works_rows      = []
topics_rows     = []
authors_rows    = []
references_rows = []

for w in all_works:
    wid       = w.get('id', '')
    title     = w.get('title', '')
    year      = w.get('publication_year')
    citations = w.get('cited_by_count', 0)
    topic     = w.get('_topic', '')
    query     = w.get('_query', '')

    # Reconstruct abstract from inverted index
    abstract = ''
    inv = w.get('abstract_inverted_index') or {}
    if inv:
        word_positions = []
        for word, positions in inv.items():
            for pos in positions:
                word_positions.append((pos, word))
        word_positions.sort()
        abstract = ' '.join(wd for _, wd in word_positions)

    works_rows.append({
        'work_id': wid, 'title': title, 'year': year,
        'cited_by_count': citations, 'topic': topic,
        'query': query, 'abstract_short': abstract[:300],
    })

    for t in w.get('topics', []) or []:
        topics_rows.append({
            'work_id': wid, 'topic': topic,
            'topic_id': t.get('id', ''),
            'topic_name': t.get('display_name', ''),
            'subfield': t.get('subfield', {}).get('display_name', ''),
            'field': t.get('field', {}).get('display_name', ''),
            'domain': t.get('domain', {}).get('display_name', ''),
            'score': t.get('score', 0),
        })

    for a in w.get('authorships', []) or []:
        author    = a.get('author', {}) or {}
        inst_list = a.get('institutions', []) or []
        inst      = inst_list[0] if inst_list else {}
        authors_rows.append({
            'work_id': wid, 'topic': topic,
            'author_id': author.get('id', ''),
            'author_name': author.get('display_name', ''),
            'institution_id': inst.get('id', ''),
            'institution_name': inst.get('display_name', ''),
            'country_code': inst.get('country_code', ''),
        })

    for ref in w.get('referenced_works', []) or []:
        references_rows.append({
            'work_id': wid, 'topic': topic, 'referenced_work_id': ref,
        })


def write_csv(rows: list, path: Path):
    if not rows:
        return
    with open(path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    print(f'  checkmark {path.name}: {len(rows)} rows')


write_csv(works_rows,      PROCESSED_DIR / 'works.csv')
write_csv(topics_rows,     PROCESSED_DIR / 'work_topics.csv')
write_csv(authors_rows,    PROCESSED_DIR / 'work_authors.csv')
write_csv(references_rows, PROCESSED_DIR / 'work_references.csv')

print('\nDownload & processing complete!')
print(f'Data saved in: {OUTPUT_DIR.resolve()}')
print('\nLevel 0 complete -> continue with Setup & Level 1')

---
## Setup — Imports & Paths

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
from pathlib import Path
from collections import Counter
from itertools import combinations

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

pd.set_option('display.max_colwidth', 100)
plt.rcParams['figure.dpi'] = 120

PROCESSED_DIR = Path('photonic_processor_data/processed')
ANALYSIS_DIR  = Path('photonic_processor_data/analysis')
ANALYSIS_DIR.mkdir(exist_ok=True)

print('Setup complete.')

---
## Level 1 — Paper Inventory
### What is in the dataset and how relevant is each paper?

### 1.1 Load Data

In [ ]:
works      = pd.read_csv(PROCESSED_DIR / 'works.csv')
topics_df  = pd.read_csv(PROCESSED_DIR / 'work_topics.csv')
authors_df = pd.read_csv(PROCESSED_DIR / 'work_authors.csv')

print('=' * 55)
print('DATASET OVERVIEW')
print('=' * 55)
print(f'  Total papers:         {len(works):>6}')
print(f'  Topic entries:        {len(topics_df):>6}')
print(f'  Author entries:       {len(authors_df):>6}')
print(f'  Year range:           {int(works["year"].min())} - {int(works["year"].max())}')
print(f'  Missing abstracts:    {works["abstract_short"].isna().sum():>6}')
print(f'  Missing titles:       {works["title"].isna().sum():>6}')

print('\n--- Papers per query ---')
print(works['query'].value_counts().to_string())

### 1.2 Data Quality Check

In [ ]:
print('=' * 55)
print('DATA QUALITY')
print('=' * 55)

works['has_abstract'] = works['abstract_short'].notna() & (works['abstract_short'].str.len() > 20)
works['has_title']    = works['title'].notna() & (works['title'].str.len() > 5)

print(f'  Papers with abstract: {works["has_abstract"].sum()} ({works["has_abstract"].mean()*100:.1f}%)')
print(f'  Papers with title:    {works["has_title"].sum()} ({works["has_title"].mean()*100:.1f}%)')

dup_titles = works[works.duplicated('title', keep=False) & works['title'].notna()]
print(f'  Duplicate titles:     {len(dup_titles)}')

### 1.3 Relevance Scoring

Each paper is scored across three keyword tiers:
- **Tier 1** (weight 3×): Core topic — directly about photonic processors
- **Tier 2** (weight 1.5×): Closely related — key components and methods
- **Tier 3** (weight 0.5×): Broader context — could be relevant or not

In [ ]:
KEYWORDS_TIER1 = [
    'photonic processor', 'photonic computing', 'optical computing',
    'photonic accelerator', 'optical accelerator', 'photonic chip',
    'silicon photonic', 'integrated photonic', 'photonic neural network',
    'optical neural network', 'photonic matrix', 'optical matrix',
    'photonic inference', 'optical inference',
]

KEYWORDS_TIER2 = [
    'waveguide', 'mach-zehnder', 'microring resonator', 'modulator',
    'photodetector', 'optical interconnect', 'coherent optical',
    'deep learning accelerator', 'neuromorphic', 'in-memory computing',
    'analog computing', 'optical switching',
]

KEYWORDS_TIER3 = [
    'photonic', 'optical', 'optics', 'laser', 'semiconductor',
    'machine learning', 'neural network', 'accelerator', 'computing',
    'FPGA', 'GPU', 'inference',
]

def compute_relevance(text: str) -> dict:
    if not isinstance(text, str):
        text = ''
    t = text.lower()
    hits1 = sum(1 for kw in KEYWORDS_TIER1 if kw in t)
    hits2 = sum(1 for kw in KEYWORDS_TIER2 if kw in t)
    hits3 = sum(1 for kw in KEYWORDS_TIER3 if kw in t)
    score = (hits1 * 3 + hits2 * 1.5 + hits3 * 0.5)
    max_score = (len(KEYWORDS_TIER1) * 3 + len(KEYWORDS_TIER2) * 1.5 + len(KEYWORDS_TIER3) * 0.5)
    norm_score = round(score / max_score, 4)
    if hits1 >= 2:
        category = 'Core Topic'
    elif hits1 == 1 or hits2 >= 2:
        category = 'Closely Related'
    elif hits2 == 1 or hits3 >= 3:
        category = 'Peripheral'
    else:
        category = 'Low Relevance'
    return {
        'relevance_score': norm_score,
        'tier1_hits': hits1,
        'tier2_hits': hits2,
        'tier3_hits': hits3,
        'relevance_category': category,
    }

works['text_combined'] = (
    works['title'].fillna('') + ' ' + works['abstract_short'].fillna('')
)
scores = works['text_combined'].apply(compute_relevance).apply(pd.Series)
works = pd.concat([works, scores], axis=1)

print('Relevance scoring complete.')
print(works['relevance_category'].value_counts())

### 1.4 Relevance Overview

In [ ]:
print('=' * 55)
print('RELEVANCE RESULTS')
print('=' * 55)

cat_counts = works['relevance_category'].value_counts()
for cat, count in cat_counts.items():
    pct = count / len(works) * 100
    print(f'  {cat:<20} {count:>5} papers  ({pct:.1f}%)')

print(f'\n  Avg relevance score:  {works["relevance_score"].mean():.3f}')
print(f'  Median score:         {works["relevance_score"].median():.3f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
order  = ['Core Topic', 'Closely Related', 'Peripheral', 'Low Relevance']
labels = [c for c in order if c in cat_counts.index]
vals   = [cat_counts.get(c, 0) for c in labels]
axes[0].pie(vals, labels=labels, colors=colors[:len(labels)], autopct='%1.1f%%', startangle=140)
axes[0].set_title('Relevance Distribution', fontsize=13, fontweight='bold')

axes[1].hist(works['relevance_score'], bins=30, color='#4C72B0', edgecolor='white')
axes[1].set_title('Score Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Relevance Score')
axes[1].set_ylabel('Number of Papers')

plt.suptitle('Photonic Processor Research — Relevance Analysis', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'step1_relevance_overview.png', bbox_inches='tight')
plt.show()

### 1.5 Relevance by Query

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
query_score = works.groupby('query')['relevance_score'].mean().sort_values()
query_score.plot(kind='barh', ax=ax, color='#4C72B0')
ax.set_title('Average Relevance Score per Query', fontsize=13, fontweight='bold')
ax.set_xlabel('Avg Score')
ax.axvline(works['relevance_score'].mean(), color='red', linestyle='--', label='Overall Avg')
ax.legend()
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'step1_relevance_by_query.png', bbox_inches='tight')
plt.show()

### 1.6 Top & Bottom Papers

In [ ]:
print('--- Top 15 Core Topic Papers (highest score) ---')
top = works[works['relevance_category'] == 'Core Topic'].nlargest(15, 'relevance_score')
print(top[['title', 'year', 'cited_by_count', 'relevance_score', 'query']].to_string(index=False))

print('\n--- Bottom 20 Low Relevance Papers (most cited) ---')
low = works[works['relevance_category'] == 'Low Relevance'].nlargest(20, 'cited_by_count')
print(low[['title', 'year', 'cited_by_count', 'query']].to_string(index=False))

### 1.7 Save Filtered Dataset

In [ ]:
relevant = works[works['relevance_category'].isin(['Core Topic', 'Closely Related'])].copy()

works.to_csv(ANALYSIS_DIR / 'step1_all_papers_scored.csv', index=False)
relevant.to_csv(ANALYSIS_DIR / 'step1_relevant_papers.csv', index=False)

print('=' * 55)
print('SAVED')
print('=' * 55)
print(f'  All papers (with score): step1_all_papers_scored.csv')
print(f'  Relevant papers only:    step1_relevant_papers.csv')
print(f'  -> {len(relevant)} of {len(works)} papers classified as relevant ({len(relevant)/len(works)*100:.1f}%)')
print('\nLevel 1 complete -> continue with Level 2: Cluster Analysis')

---
## Level 2 — Cluster Analysis
### Grouping the relevant papers into subfields

### 2.1 Load Relevant Papers

In [ ]:
works = pd.read_csv(ANALYSIS_DIR / 'step1_relevant_papers.csv')

print(f'Relevant papers loaded: {len(works)}')
print(f'Columns: {list(works.columns)}')

### 2.2 Prepare Text for Clustering
Title is doubled to give it stronger signal weight.

In [ ]:
works['text_for_clustering'] = (
    works['title'].fillna('') + ' ' +
    works['title'].fillna('') + ' ' +
    works['abstract_short'].fillna('')
)

works = works[works['text_for_clustering'].str.len() > 30].copy()
print(f'Papers with sufficient text: {len(works)}')

### 2.3 TF-IDF Vectorization

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.85,
    stop_words='english',
    sublinear_tf=True,
)

X = vectorizer.fit_transform(works['text_for_clustering'])
print(f'TF-IDF matrix: {X.shape[0]} papers x {X.shape[1]} features')

### 2.4 Find Optimal Number of Clusters
Testing k=4 to k=12 using Silhouette Score.

In [ ]:
print('Searching for optimal cluster count ...')

svd = TruncatedSVD(n_components=50, random_state=42)
X_reduced = svd.fit_transform(X)
X_norm = normalize(X_reduced)

k_range = range(4, 13)
silhouette_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = km.fit_predict(X_norm)
    score = silhouette_score(X_norm, labels, sample_size=min(1000, len(works)))
    silhouette_scores.append(score)
    print(f'  k={k:>2}: Silhouette = {score:.4f}')

best_k = k_range.start + silhouette_scores.index(max(silhouette_scores))
print(f'\n-> Best k = {best_k} (Score: {max(silhouette_scores):.4f})')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(k_range), silhouette_scores, marker='o', color='#4C72B0', linewidth=2)
ax.axvline(best_k, color='red', linestyle='--', label=f'Best k={best_k}')
ax.set_title('Silhouette Score — Optimal Cluster Count', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Silhouette Score')
ax.legend()
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'step2_silhouette_scores.png', bbox_inches='tight')
plt.show()

### 2.5 Cluster Papers

In [ ]:
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=20, max_iter=500)
works['cluster_id'] = km_final.fit_predict(X_norm)

print('Cluster distribution:')
print(works['cluster_id'].value_counts().sort_index().to_string())

### 2.6 Auto-Label Clusters by Top Keywords

In [ ]:
SUBFIELD_RULES = [
    ('Hardware & Chip Design',        ['waveguide', 'fabrication', 'mach-zehnder', 'modulator',
                                       'ring resonator', 'silicon photonic', 'integrated circuit',
                                       'cmos', 'chip', 'foundry', 'nanophotonic']),
    ('Optical Neural Networks',       ['neural network', 'deep learning', 'convolutional', 'training',
                                       'inference', 'classification', 'backpropagation', 'layer',
                                       'activation', 'weight', 'optical neural']),
    ('Matrix & Linear Algebra',       ['matrix multiplication', 'matrix vector', 'linear algebra',
                                       'dot product', 'crossbar', 'analog', 'weight matrix',
                                       'vector', 'multiplication']),
    ('Photonic Accelerators',         ['accelerator', 'throughput', 'latency', 'energy efficiency',
                                       'hardware accelerator', 'dnn', 'inference engine',
                                       'transformer', 'attention']),
    ('Interconnects & Communication', ['interconnect', 'bandwidth', 'data center', 'optical link',
                                       'transceiver', 'wdm', 'wavelength', 'transmission',
                                       'network on chip', 'noc']),
    ('Phase Change & Memory',         ['phase change', 'pcm', 'memristor', 'nonvolatile',
                                       'memory', 'gst', 'in-memory', 'analog memory', 'resistive']),
    ('Quantum & Sensing',             ['quantum', 'qubit', 'entanglement', 'sensing',
                                       'lidar', 'imaging', 'photon', 'detector']),
    ('Simulation & Modeling',         ['simulation', 'modeling', 'fdtd', 'finite element',
                                       'numerical', 'algorithm', 'optimization', 'framework']),
]

def assign_subfield(top_words: list) -> str:
    best = ('Other', 0)
    for subfield, keywords in SUBFIELD_RULES:
        hits = sum(1 for kw in keywords if any(kw in w for w in top_words))
        if hits > best[1]:
            best = (subfield, hits)
    return best[0]

cluster_info = {}
print('=== Top Keywords per Cluster ===')

for cid in sorted(works['cluster_id'].unique()):
    cluster_texts = works[works['cluster_id'] == cid]['text_for_clustering'].tolist()
    sub_vec = TfidfVectorizer(max_features=500, ngram_range=(1,2),
                               stop_words='english', sublinear_tf=True)
    try:
        sub_X = sub_vec.fit_transform(cluster_texts)
        mean_tfidf = np.asarray(sub_X.mean(axis=0)).flatten()
        top_idx = mean_tfidf.argsort()[-15:][::-1]
        top_words = [sub_vec.get_feature_names_out()[i] for i in top_idx]
    except Exception:
        top_words = []

    subfield = assign_subfield(top_words)
    n_papers = len(works[works['cluster_id'] == cid])
    cluster_info[cid] = {'subfield': subfield, 'top_words': top_words[:10], 'n_papers': n_papers}

    print(f'\n  Cluster {cid} -> "{subfield}" ({n_papers} papers)')
    print(f'  Keywords: {", ".join(top_words[:10])}')

works['subfield'] = works['cluster_id'].map(lambda cid: cluster_info[cid]['subfield'])

### 2.7 Visualize Clusters

In [ ]:
subfield_counts = works['subfield'].value_counts()

# Chart 1: Papers per subfield
fig, ax = plt.subplots(figsize=(10, 5))
colors = plt.cm.tab10(np.linspace(0, 1, len(subfield_counts)))
bars = ax.barh(subfield_counts.index, subfield_counts.values, color=colors)
ax.bar_label(bars, padding=4, fontsize=10)
ax.set_title('Papers per Subfield', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Papers')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'step2_papers_per_subfield.png', bbox_inches='tight')
plt.show()

# Chart 2: PCA scatter
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_norm)

fig, ax = plt.subplots(figsize=(11, 7))
palette = plt.cm.tab10(np.linspace(0, 1, best_k))

for cid in sorted(works['cluster_id'].unique()):
    mask = works['cluster_id'] == cid
    label = cluster_info[cid]['subfield']
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               color=palette[cid], label=f'C{cid}: {label}',
               alpha=0.6, s=18, linewidths=0)

ax.set_title('Cluster Distribution in 2D Space (PCA)', fontsize=14, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.legend(loc='upper right', fontsize=8, markerscale=2)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'step2_cluster_scatter.png', bbox_inches='tight')
plt.show()

In [ ]:
# Chart 3: Subfields over time
year_subfield = works.groupby(['year', 'subfield']).size().unstack(fill_value=0)
year_subfield = year_subfield[year_subfield.index.astype(int).map(lambda y: 2010 <= y <= 2025)]

fig, ax = plt.subplots(figsize=(13, 6))
year_subfield.plot(kind='area', stacked=True, ax=ax, colormap='tab10', alpha=0.85)
ax.set_title('Subfield Growth Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Papers')
ax.legend(loc='upper left', fontsize=8, bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'step2_subfields_over_time.png', bbox_inches='tight')
plt.show()

# Chart 4: Citation heatmap
citation_stats = works.groupby('subfield')['cited_by_count'].agg(['mean', 'median', 'max'])
citation_stats.columns = ['Avg Citations', 'Median', 'Max']
citation_stats = citation_stats.sort_values('Avg Citations', ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(citation_stats, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Citation Statistics per Subfield', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'step2_citation_heatmap.png', bbox_inches='tight')
plt.show()

### 2.8 Top Papers per Subfield

In [ ]:
print('=' * 60)
print('TOP 3 PAPERS PER SUBFIELD (by citations)')
print('=' * 60)

for subfield in works['subfield'].unique():
    sub_df = works[works['subfield'] == subfield]
    top3 = sub_df.nlargest(3, 'cited_by_count')[['title', 'year', 'cited_by_count']]
    print(f'\n--- {subfield} ({len(sub_df)} papers) ---')
    print(top3.to_string(index=False))

### 2.9 Save Clustered Dataset

In [ ]:
works.to_csv(ANALYSIS_DIR / 'step2_papers_clustered.csv', index=False)

summary_rows = []
for cid, info in cluster_info.items():
    summary_rows.append({
        'cluster_id': cid,
        'subfield': info['subfield'],
        'n_papers': info['n_papers'],
        'top_keywords': ', '.join(info['top_words']),
    })
pd.DataFrame(summary_rows).to_csv(ANALYSIS_DIR / 'step2_cluster_summary.csv', index=False)

print('=' * 60)
print('SAVED')
print('=' * 60)
print('  step2_papers_clustered.csv  — all papers with subfield')
print('  step2_cluster_summary.csv   — cluster overview')
print(f'\n  {best_k} subfields found:')
for sf, count in subfield_counts.items():
    print(f'    {sf:<35} {count:>4} papers')
print('\nLevel 2 complete -> continue with Level 3: Field Architecture')

---
## Level 3 — Field Architecture
### Which subfields are connected? Where are the bridges? Is the field cohesive or fragmented?

### 3.1 Load Data

In [ ]:
works      = pd.read_csv(ANALYSIS_DIR / 'step2_papers_clustered.csv')
references = pd.read_csv(PROCESSED_DIR / 'work_references.csv')

print(f'Papers loaded:      {len(works)}')
print(f'References loaded:  {len(references)}')
print(f'Subfields:          {works["subfield"].nunique()}')
print(works['subfield'].value_counts().to_string())

### 3.2 Cross-Subfield Citation Links

In [ ]:
id_to_subfield = works.set_index('work_id')['subfield'].to_dict()

refs_internal = references[
    references['work_id'].isin(id_to_subfield) &
    references['referenced_work_id'].isin(id_to_subfield)
].copy()

refs_internal['subfield_from'] = refs_internal['work_id'].map(id_to_subfield)
refs_internal['subfield_to']   = refs_internal['referenced_work_id'].map(id_to_subfield)
refs_cross = refs_internal[refs_internal['subfield_from'] != refs_internal['subfield_to']].copy()

print(f'Internal references total:          {len(refs_internal)}')
print(f'Cross-subfield references:          {len(refs_cross)}')

edge_counts = refs_cross.groupby(['subfield_from', 'subfield_to']).size().reset_index(name='weight')

edge_sym = {}
for _, row in edge_counts.iterrows():
    key = tuple(sorted([row['subfield_from'], row['subfield_to']]))
    edge_sym[key] = edge_sym.get(key, 0) + row['weight']

edges_df = pd.DataFrame(
    [(a, b, w) for (a, b), w in edge_sym.items()],
    columns=['source', 'target', 'weight']
).sort_values('weight', ascending=False)

print('\n=== Strongest Connections Between Subfields ===')
print(edges_df.to_string(index=False))

### 3.3 Textual Similarity Between Subfields

In [ ]:
subfields = works['subfield'].unique()

subfield_texts = (
    works.groupby('subfield')['text_combined']
    .apply(lambda x: ' '.join(x.fillna('')))
)

vec = TfidfVectorizer(max_features=2000, ngram_range=(1,2), stop_words='english', sublinear_tf=True)
S = vec.fit_transform(subfield_texts)
sim_matrix = cosine_similarity(S)
sim_df = pd.DataFrame(sim_matrix, index=subfield_texts.index, columns=subfield_texts.index)

print('=== Textual Similarity Between Subfields (Cosine) ===')
print(sim_df.round(3).to_string())

### 3.4 Build Network & Compute Centrality

In [ ]:
G = nx.Graph()
sf_counts = works['subfield'].value_counts()

for sf in subfields:
    G.add_node(sf, n_papers=sf_counts[sf])

max_w = edges_df['weight'].max()
for _, row in edges_df.iterrows():
    G.add_edge(row['source'], row['target'],
               weight=row['weight'],
               weight_norm=row['weight'] / max_w)

print('=== Network Metrics ===')
print(f'  Nodes:      {G.number_of_nodes()}')
print(f'  Edges:      {G.number_of_edges()}')
print(f'  Density:    {nx.density(G):.3f}')
if nx.is_connected(G):
    print(f'  Connected:  Fully connected')
    print(f'  Diameter:   {nx.diameter(G)}')
else:
    components = list(nx.connected_components(G))
    print(f'  Connected:  FRAGMENTED ({len(components)} components)')

degree_centrality = nx.degree_centrality(G)
betweenness       = nx.betweenness_centrality(G, weight='weight')
eigenvector       = nx.eigenvector_centrality(G, weight='weight', max_iter=500)

centrality_df = pd.DataFrame({
    'subfield':    list(degree_centrality.keys()),
    'degree':      list(degree_centrality.values()),
    'betweenness': [betweenness[sf] for sf in degree_centrality],
    'eigenvector': [eigenvector[sf] for sf in degree_centrality],
    'n_papers':    [G.nodes[sf]['n_papers'] for sf in degree_centrality],
}).sort_values('betweenness', ascending=False)

print('\n=== Centrality per Subfield ===')
print(centrality_df.round(4).to_string(index=False))

### 3.5 Field Architecture Network

In [ ]:
fig, ax = plt.subplots(figsize=(13, 9))
pos = nx.spring_layout(G, weight='weight', seed=42, k=2.5)

node_sizes  = [G.nodes[n]['n_papers'] * 1.8 for n in G.nodes]
edge_weights = [G[u][v]['weight_norm'] * 8 for u, v in G.edges]
color_vals  = [betweenness[n] for n in G.nodes]
norm_c      = plt.Normalize(min(color_vals), max(color_vals))
node_colors = [cm.RdYlGn(norm_c(v)) for v in color_vals]

nx.draw_networkx_edges(G, pos, ax=ax, width=edge_weights, alpha=0.55, edge_color='#555555')
nx.draw_networkx_nodes(G, pos, ax=ax, node_size=node_sizes, node_color=node_colors, alpha=0.92)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=9, font_weight='bold',
                        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.75, ec='none'))

edge_labels = {(u, v): int(G[u][v]['weight']) for u, v in G.edges}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax, font_size=7, alpha=0.7)

sm = cm.ScalarMappable(cmap=cm.RdYlGn, norm=norm_c)
sm.set_array([])
plt.colorbar(sm, ax=ax, label='Betweenness Centrality', shrink=0.6, pad=0.02)

ax.set_title('Field Architecture: Photonic Processor Research\n'
             'Node size = number of papers  |  Edge width = citation connections  |  '
             'Color = bridge centrality',
             fontsize=12, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'step3_field_architecture.png', bbox_inches='tight', dpi=150)
plt.show()

### 3.6 Textual Similarity Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
mask = np.eye(len(sim_df), dtype=bool)
sns.heatmap(sim_df, annot=True, fmt='.2f', cmap='Blues', mask=mask,
            linewidths=0.5, ax=ax, vmin=0, vmax=1)
ax.set_title('Textual Similarity Between Subfields\n(TF-IDF Cosine Similarity)',
             fontsize=12, fontweight='bold')
plt.xticks(rotation=35, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'step3_similarity_heatmap.png', bbox_inches='tight')
plt.show()

### 3.7 Architecture Diagnosis

In [ ]:
print('=' * 60)
print('FIELD ARCHITECTURE DIAGNOSIS')
print('=' * 60)

density  = nx.density(G)
avg_sim  = sim_df.values[~np.eye(len(sim_df), dtype=bool)].mean()
bridge_node  = centrality_df.iloc[0]['subfield']
bridge_score = centrality_df.iloc[0]['betweenness']
largest_sf   = works['subfield'].value_counts().index[0]
largest_n    = works['subfield'].value_counts().iloc[0]

print(f'\n  Network density:          {density:.3f}  ', end='')
print('(dense)' if density > 0.6 else '(medium)' if density > 0.3 else '(sparse/fragmented)')

print(f'  Avg textual similarity:   {avg_sim:.3f}  ', end='')
print('(cohesive)' if avg_sim > 0.4 else '(medium)' if avg_sim > 0.2 else '(heterogeneous)')

print(f'\n  Dominant subfield:        {largest_sf} ({largest_n} papers)')
print(f'  Bridge subfield:          {bridge_node} (Betweenness: {bridge_score:.3f})')

print('\n  Strongest connections:')
for _, row in edges_df.head(4).iterrows():
    print(f'    {row["source"]} <-> {row["target"]}: {int(row["weight"])} citations')

print('\n  Weakest connections:')
for _, row in edges_df.tail(3).iterrows():
    print(f'    {row["source"]} <-> {row["target"]}: {int(row["weight"])} citations')

### 3.8 Save

In [ ]:
centrality_df.to_csv(ANALYSIS_DIR / 'step3_centrality.csv', index=False)
edges_df.to_csv(ANALYSIS_DIR / 'step3_edges.csv', index=False)
sim_df.to_csv(ANALYSIS_DIR / 'step3_similarity_matrix.csv')

print('Saved: step3_centrality.csv, step3_edges.csv, step3_similarity_matrix.csv')
print('\nLevel 3 complete -> continue with Level 4: Field-Engineering Gaps')

---
## Level 4 — Field-Engineering Gaps
### Where are missing connections? Where is a function missing? Where exist building blocks but no emergent field?

### 4.1 Load Data

In [ ]:
works      = pd.read_csv(ANALYSIS_DIR / 'step2_papers_clustered.csv')
edges_df   = pd.read_csv(ANALYSIS_DIR / 'step3_edges.csv')
centrality = pd.read_csv(ANALYSIS_DIR / 'step3_centrality.csv')
sim_df     = pd.read_csv(ANALYSIS_DIR / 'step3_similarity_matrix.csv', index_col=0)

subfields = works['subfield'].unique().tolist()
sf_counts = works['subfield'].value_counts()

print(f'Papers: {len(works)}  |  Subfields: {len(subfields)}')

### 4.2 Gap Matrix
Expected vs. actual connection strength using a gravity model.
Gap = positive → underconnected; negative → overconnected.

In [ ]:
weight_matrix = pd.DataFrame(0.0, index=subfields, columns=subfields)
for _, row in edges_df.iterrows():
    weight_matrix.loc[row['source'], row['target']] = row['weight']
    weight_matrix.loc[row['target'], row['source']] = row['weight']

total_papers = len(works)
expected_matrix = pd.DataFrame(0.0, index=subfields, columns=subfields)
for sf_a in subfields:
    for sf_b in subfields:
        if sf_a != sf_b:
            expected_matrix.loc[sf_a, sf_b] = (sf_counts[sf_a] * sf_counts[sf_b]) / total_papers

gap_matrix = pd.DataFrame(0.0, index=subfields, columns=subfields)
for sf_a in subfields:
    for sf_b in subfields:
        if sf_a != sf_b:
            exp = expected_matrix.loc[sf_a, sf_b]
            act = weight_matrix.loc[sf_a, sf_b]
            gap_matrix.loc[sf_a, sf_b] = (exp - act) / exp if exp > 0 else 0

gap_sym = (gap_matrix + gap_matrix.T) / 2

print('=== Gap Matrix (positive = underconnected) ===')
print(gap_sym.round(3).to_string())

### 4.3 Growth Momentum per Subfield

In [ ]:
works['year'] = pd.to_numeric(works['year'], errors='coerce')
recent = works[works['year'] >= 2020]
older  = works[works['year'].between(2015, 2019)]

growth = pd.DataFrame({
    'subfield':      subfields,
    'papers_total':  [sf_counts.get(sf, 0) for sf in subfields],
    'papers_recent': [len(recent[recent['subfield'] == sf]) for sf in subfields],
    'papers_older':  [len(older[older['subfield'] == sf]) for sf in subfields],
})
growth['growth_ratio'] = growth.apply(
    lambda r: r['papers_recent'] / r['papers_older'] if r['papers_older'] > 0 else float('inf'),
    axis=1
)
growth['citations_avg'] = [
    works[works['subfield'] == sf]['cited_by_count'].mean() for sf in subfields
]
growth = growth.sort_values('growth_ratio', ascending=False)

print('=== Growth Momentum per Subfield ===')
print(growth.to_string(index=False))

### 4.4 Concept Gaps
Keywords present in subfield A but absent in underconnected subfield B.

In [ ]:
subfield_texts = (
    works.groupby('subfield')['text_combined']
    .apply(lambda x: ' '.join(x.fillna('')))
)
vec = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), stop_words='english', sublinear_tf=True)
S = vec.fit_transform(subfield_texts)
feature_names = vec.get_feature_names_out()

sf_keywords = {}
for i, sf in enumerate(subfield_texts.index):
    row = np.asarray(S[i].todense()).flatten()
    top_idx = row.argsort()[-30:][::-1]
    sf_keywords[sf] = set(feature_names[top_idx])

print('=== Concept Gaps Between Underconnected Subfields ===')

gap_pairs = []
for sf_a in subfields:
    for sf_b in subfields:
        if sf_a >= sf_b:
            continue
        gap_score = gap_sym.loc[sf_a, sf_b]
        if gap_score > 0.3:
            missing_in_b = sf_keywords.get(sf_a, set()) - sf_keywords.get(sf_b, set())
            missing_in_a = sf_keywords.get(sf_b, set()) - sf_keywords.get(sf_a, set())
            gap_pairs.append({
                'subfield_a': sf_a,
                'subfield_b': sf_b,
                'gap_score': round(gap_score, 3),
                'concepts_a_missing_in_b': ', '.join(sorted(missing_in_b)[:8]),
                'concepts_b_missing_in_a': ', '.join(sorted(missing_in_a)[:8]),
            })
            print(f'\n  {sf_a} <-> {sf_b}  (Gap: {gap_score:.3f})')
            print(f'    "{sf_a}" knows, "{sf_b}" does not: {", ".join(sorted(missing_in_b)[:6])}')
            print(f'    "{sf_b}" knows, "{sf_a}" does not: {", ".join(sorted(missing_in_a)[:6])}')

gaps_df = pd.DataFrame(gap_pairs) if gap_pairs else pd.DataFrame()
if gaps_df.empty:
    print('  No major concept gaps found (all subfields are well-connected).')

### 4.5 Visualizations

In [ ]:
fig = plt.figure(figsize=(18, 14))
fig.suptitle('Field-Engineering Gaps: Photonic Processor Research',
             fontsize=15, fontweight='bold', y=1.01)

# Chart 1: Gap Heatmap
ax1 = fig.add_subplot(2, 2, 1)
mask = np.eye(len(gap_sym), dtype=bool)
sns.heatmap(gap_sym, annot=True, fmt='.2f', cmap='RdYlGn_r', mask=mask,
            linewidths=0.5, ax=ax1, vmin=-0.5, vmax=1.0,
            cbar_kws={'label': 'Gap Score\n(red = underconnected)'})
ax1.set_title('Connection Gaps Between Subfields', fontweight='bold')
ax1.tick_params(axis='x', rotation=35)
ax1.tick_params(axis='y', rotation=0)

# Chart 2: Bubble Chart — Growth vs Citations
ax2 = fig.add_subplot(2, 2, 2)
colors_bubble = plt.cm.tab10(np.linspace(0, 1, len(growth)))
for i, (_, row) in enumerate(growth.iterrows()):
    ax2.scatter(row['growth_ratio'], row['citations_avg'],
                s=row['papers_total'] * 0.8, color=colors_bubble[i], alpha=0.85, zorder=3)
    ax2.annotate(row['subfield'], (row['growth_ratio'], row['citations_avg']),
                 textcoords='offset points', xytext=(6, 4), fontsize=8, fontweight='bold')
ax2.set_xlabel('Growth Rate (papers 2020-2025 / 2015-2019)')
ax2.set_ylabel('Avg Citations')
ax2.set_title('Momentum: Growth vs. Impact\n(bubble size = number of papers)', fontweight='bold')
ax2.axvline(1.0, color='gray', linestyle='--', alpha=0.5, label='No growth')
ax2.axhline(works['cited_by_count'].mean(), color='orange',
            linestyle='--', alpha=0.5, label='Overall avg citations')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# Chart 3: Connection Network (real citation weights)
ax3 = fig.add_subplot(2, 2, 3)
G_gap = nx.Graph()
for sf in subfields:
    G_gap.add_node(sf, n_papers=sf_counts[sf])
for _, row in edges_df.iterrows():
    G_gap.add_edge(row['source'], row['target'], weight=row['weight'])

pos = nx.spring_layout(G_gap, seed=42, k=3)
node_sizes_g = [G_gap.nodes[n]['n_papers'] * 1.5 for n in G_gap.nodes]
all_weights  = [G_gap[u][v]['weight'] for u, v in G_gap.edges]
max_w = max(all_weights)
min_w = min(all_weights)

edge_colors = [cm.RdYlGn((G_gap[u][v]['weight'] - min_w) / (max_w - min_w)) for u, v in G_gap.edges]
edge_widths = [1.5 + 6 * (G_gap[u][v]['weight'] - min_w) / (max_w - min_w) for u, v in G_gap.edges]
edge_labels_net = {(u, v): str(int(G_gap[u][v]['weight'])) for u, v in G_gap.edges}

nx.draw_networkx_edges(G_gap, pos, ax=ax3, edge_color=edge_colors, width=edge_widths, alpha=0.85)
nx.draw_networkx_nodes(G_gap, pos, ax=ax3, node_size=node_sizes_g, node_color='#4C72B0', alpha=0.9)
nx.draw_networkx_labels(G_gap, pos, ax=ax3, font_size=8, font_weight='bold',
                        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.85, ec='none'))
nx.draw_networkx_edge_labels(G_gap, pos, edge_labels=edge_labels_net, ax=ax3, font_size=6, alpha=0.8)

red_patch   = mpatches.Patch(color=cm.RdYlGn(0.0), label='Weak connection')
green_patch = mpatches.Patch(color=cm.RdYlGn(1.0), label='Strong connection')
ax3.legend(handles=[red_patch, green_patch], fontsize=7, loc='lower right')
ax3.set_title('Connection Network\n(green/thick = strong, red/thin = weak)', fontweight='bold')
ax3.axis('off')

# Chart 4: Publication Trend
ax4 = fig.add_subplot(2, 2, 4)
year_sf = works[works['year'].between(2015, 2025)].groupby(['year', 'subfield']).size().unstack(fill_value=0)
year_sf.plot(kind='line', ax=ax4, marker='o', markersize=4, linewidth=2)
ax4.set_title('Publication Trend 2015-2025', fontweight='bold')
ax4.set_xlabel('Year')
ax4.set_ylabel('Number of Papers')
ax4.legend(fontsize=7, loc='upper left')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'step4_field_gaps.png', bbox_inches='tight', dpi=150)
plt.show()

### 4.6 Gap Report — Strategic Questions

In [ ]:
print('=' * 65)
print('FIELD-ENGINEERING GAP REPORT')
print('=' * 65)

print('\n1. WHERE ARE MISSING CONNECTIONS?')
print('   (Subfield pairs that cite each other less than expected given their size)')
top_gaps = gap_sym.copy()
np.fill_diagonal(top_gaps.values, 0)
stack = top_gaps.stack().sort_values(ascending=False)
positive_gaps = stack[stack > 0]
if len(positive_gaps) > 0:
    seen = set()
    for (a, b), gap in positive_gaps.items():
        key = tuple(sorted([a, b]))
        if key not in seen:
            seen.add(key)
            print(f'   -> {a} <-> {b}: Gap Score {gap:.3f}')
        if len(seen) >= 6:
            break
else:
    print('   -> No underconnected pairs found. All subfields are well-connected.')

print('\n2. WHERE IS A FUNCTION MISSING?')
print('   (High growth subfields with low bridge centrality)')
low_bridge = centrality[centrality['betweenness'] < 0.1].merge(
    growth[['subfield', 'growth_ratio']], on='subfield'
).sort_values('growth_ratio', ascending=False)
for _, row in low_bridge.iterrows():
    print(f'   -> {row["subfield"]}: growing {row["growth_ratio"]:.1f}x '
          f'but Betweenness={row["betweenness"]:.3f} -> isolated growth')

print('\n3. WHERE EXIST BUILDING BLOCKS BUT NO EMERGENT FIELD?')
print('   (High textual similarity + low citation connection)')
found_any = False
for sf_a in subfields:
    for sf_b in subfields:
        if sf_a >= sf_b:
            continue
        sim  = sim_df.loc[sf_a, sf_b]
        gap  = gap_sym.loc[sf_a, sf_b]
        if sim > 0.55 and gap > 0.3:
            print(f'   -> {sf_a} & {sf_b}: '
                  f'Textually similar ({sim:.2f}) but underconnected (gap {gap:.2f})'
                  f' -> Fusion potential')
            found_any = True
if not found_any:
    print('   -> No clear fusion candidates. The field is already well-integrated.')

print('\n4. STRATEGIC RECOMMENDATIONS')
fastest = growth.iloc[0]['subfield']
slowest = growth.iloc[-1]['subfield']
bridge  = centrality.sort_values('betweenness', ascending=False).iloc[0]['subfield']

gap_copy = gap_sym.copy()
np.fill_diagonal(gap_copy.values, -999)
max_idx = np.unravel_index(gap_copy.values.argmax(), gap_copy.shape)
sf_a_gap = gap_copy.index[max_idx[0]]
sf_b_gap = gap_copy.columns[max_idx[1]]

print(f'   -> Invest in "{fastest}": highest growth rate, '
      f'strategically important for scaling')
print(f'   -> Study "{bridge}": bridge subfield, '
      f'connects otherwise isolated areas')
print(f'   -> Close gap: {sf_a_gap} <-> {sf_b_gap} '
      f'— weakest relative connection, needs interdisciplinary research')
print(f'   -> Monitor "{slowest}": lowest growth, '
      f'possibly a mature or stagnating subfield')

### 4.7 Save

In [ ]:
growth.to_csv(ANALYSIS_DIR / 'step4_growth_momentum.csv', index=False)
gap_sym.to_csv(ANALYSIS_DIR / 'step4_gap_matrix.csv')
if not gaps_df.empty:
    gaps_df.to_csv(ANALYSIS_DIR / 'step4_concept_gaps.csv', index=False)

print('=' * 65)
print('SAVED')
print('=' * 65)
print('  step4_growth_momentum.csv   — growth & impact per subfield')
print('  step4_gap_matrix.csv        — gap scores between subfields')
print('  step4_concept_gaps.csv      — concept gaps (if any found)')
print('\n✅ All 4 levels complete!')
print(f'   Outputs saved in: {ANALYSIS_DIR.resolve()}')